# 04 · Generate surfaces and an interface

## Objective

This notebook generates slabs from bulk LiNbO3 and SiO2 structures. It selects the surface terminations and match the in-plane lattices, and builds one interface and saves it for geometry optimisation in Notebook 05.

It uses `SlabFactory` and `InterfaceFactory` implementations to create interfaces, and inspects the slabs and interface with the 3D viewer. This notebook generates a starting geometry. It does not relax the structure or calculate energies.

## Prerequisites
Before execution, make sure these dependencies are met,

Install the notebook dependencies:

   ```bash
   uv sync --group notebooks
   ```

## 1. Import the generation and viewing tools

This step imports the structure factories, termination filter, and 3D viewer. It imports ASE to write and read the interface file, and `jsanitize` to store interface metadata as ordinary numeric lists.

In [ ]:
import py3Dmol
from ase.io import read, write
from IPython.display import Markdown
from IPython.display import display as show
from monty.json import jsanitize
from pymatgen.core.structure import Structure
from pymatgen.io.ase import AseAtomsAdaptor

from crystal import CRYSTAL_ROOT
from crystal.structure.interfaces.builder import (
    Interface,
    InterfaceFactory,
    InterfaceMatchPolicy,
)
from crystal.structure.slabs.builder import (
    SlabFactory,
    TerminationLayer,
    filter_slabs_by_termination,
)


## 2. Set the input paths and slab parameters

This step selects LiNbO3 as the top material and SiO2 as the bottom material. It sets a `1×1×2` supercell, the `(0, 0, 1)` surface orientation, and an initial slab vacuum of two layer units for each material.

This step sets the top termination selector to `Li_`. The generation step selects the first matching top slab and the first bottom slab. It prints the available terminations for inspection.

This step sets `INTERFACE_PATH` to the file used by Notebook 05. The export cell overwrites this file on each run.

In [ ]:
DATA_DIR = CRYSTAL_ROOT / "examples/results"
INTERFACE_PATH = DATA_DIR / "interface.extxyz"
TOP_BULK_PATH = CRYSTAL_ROOT / "tests/data/relaxed_cifs_cell/mp3731_cell.cif"
BOTTOM_BULK_PATH = CRYSTAL_ROOT / "tests/data/relaxed_cifs_cell/mp8352_cell.cif"

SUPERCELL = (1, 1, 2)
MILLER_INDICES = (0, 0, 1)
VACUUM_LAYERS = 2
TOP_TERMINATION = "Li_"
TOP_TERMINATION_LAYER = TerminationLayer.top


## 3. Generate slabs and select terminations

This step reads the bulk structures and generates slab candidates. It prints each candidate index and its top-to-bottom termination labels. Candidates can have different layer arrangements even when their surface labels are the same.

This step applies the top termination filter and checks that both materials have a suitable slab. It keeps the first selected slab from each material for lattice matching.

In [ ]:
top_bulk = Structure.from_file(TOP_BULK_PATH)
bottom_bulk = Structure.from_file(BOTTOM_BULK_PATH)
top_slab_candidates = SlabFactory(
    structure=top_bulk,
    supercell=SUPERCELL,
    vacuum_thickness_in_layers=VACUUM_LAYERS,
    surface_miller_indices=MILLER_INDICES,
).generate_slabs()
bottom_slab_candidates = SlabFactory(
    structure=bottom_bulk,
    supercell=SUPERCELL,
    vacuum_thickness_in_layers=VACUUM_LAYERS,
    surface_miller_indices=MILLER_INDICES,
).generate_slabs()

for label, slabs in [
    ("LiNbO3", top_slab_candidates),
    ("SiO2", bottom_slab_candidates),
]:
    print(f"{label}: {len(slabs)} slab candidates")
    for index, slab in enumerate(slabs):
        print(f"  [{index}] {slab.top_layer_termination} -> {slab.bottom_layer_termination}")

filtered_top_slabs = filter_slabs_by_termination(
    slabs=top_slab_candidates,
    termination=TOP_TERMINATION,
    at_layer=TOP_TERMINATION_LAYER,
)
if not filtered_top_slabs or not bottom_slab_candidates:
    msg = f"No suitable slabs for top termination {TOP_TERMINATION}. Check the inputs and slab settings."
    raise RuntimeError(msg)
top_slab = filtered_top_slabs[0]
bottom_slab = bottom_slab_candidates[0]
print("Selected top/bottom slab atoms:", len(top_slab), len(bottom_slab))


## 4. Inspect the selected slabs in 3D (optional)

This step defines a shared viewer and shows the selected slabs as `2×2×1` periodic copies with unit-cell boundaries. Only copies are changed; the calculation inputs remain unchanged.

Drag to rotate a structure. Scroll to zoom. Run this cell if you also want to use the final interface views.

In [ ]:
def show_structure(
    structure: Structure,
    title: str,
    *,
    repeat: tuple[int, int, int] = (1, 1, 1),
) -> None:
    view_structure = structure.copy()
    view_structure.make_supercell(repeat)
    viewer = py3Dmol.view(width=620, height=420)
    viewer.addModel(view_structure.to(fmt="cif"), "cif")
    viewer.setStyle({"stick": {"radius": 0.16}, "sphere": {"scale": 0.28}})
    viewer.addUnitCell()
    viewer.zoomTo()
    show(Markdown(f"### {title}"))
    show(viewer)


show_structure(top_slab, "Selected top slab (2×2×1)", repeat=(2, 2, 1))
show_structure(bottom_slab, "Selected bottom slab (2×2×1)", repeat=(2, 2, 1))


## 5. Set the interface matching parameters

This step defines the lattice-matching policy. It limits the matched in-plane area to 600 Å² and sets the mismatch tolerances to 0.1. `InterfaceFactory` makes the slab lattices orthogonal in plane before matching.

This step sets the interface vacuum to 15 Å. The generation step uses `gap=None` so that the factory estimates the initial separation. This notebook retains the first valid match. Notebook 05 uses that match to compare interface offsets.

In [ ]:
INTERFACE_VACUUM_ANGSTROM = 15.0
match_policy = InterfaceMatchPolicy(
    maximum_area_ratio_tolerance=0.1,
    maximum_area_in_angstrom_square=600.0,
    length_tolerance_in_angstrom=0.1,
    angle_tolerance_in_radian=0.1,
)


## 6. Match the lattices and build an interface

This step builds the interface factory from the selected slabs and matching policy. It generates the first matched interface and checks that a match exists and its atoms do not overlap. It stops if either check fails.

This step retains the interface metadata. The metadata identifies the top and bottom slabs and lets later notebooks reconstruct the interface.

In [ ]:
factory = InterfaceFactory(
    top_slab=top_slab,
    bottom_slab=bottom_slab,
    interface_match_policy=match_policy,
)
generated_interface = next(
    factory.generate_interfaces(gap=None, vacuum=INTERFACE_VACUUM_ANGSTROM),
    None,
)
if generated_interface is None:
    msg = f"No interface match was found with an area limit of {match_policy.maximum_area_in_angstrom_square} Å². Check the matching settings."
    raise RuntimeError(msg)
if not generated_interface.is_valid():
    msg = "The generated interface contains overlapping atoms. Check the selected slabs and gap."
    raise RuntimeError(msg)
print("Generated interface atoms:", len(generated_interface))
print("Interface valid:", generated_interface.is_valid())


## 7. Save and read back the interface

This step converts the interface to ASE atoms, normalises its metadata with `jsanitize`, and writes `results/interface.extxyz` for Notebook 05. The extxyz header stores the interface metadata. The per-atom `interface_label` field identifies the film and substrate atoms.

This step reads the file back, reconstructs the `Interface` object, and prints the total atom count and matched slab atom counts. This checks that the saved file can be used by the next notebook. The file is overwritten each time this cell runs.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
ase_atoms = AseAtomsAdaptor.get_atoms(generated_interface)
ase_atoms.info = jsanitize(ase_atoms.info, strict=True)
write(INTERFACE_PATH.as_posix(), ase_atoms, format="extxyz")

saved_atoms = read(INTERFACE_PATH.as_posix(), format="extxyz")
saved_atoms.info = jsanitize(saved_atoms.info, strict=True)
interface = Interface.from_structure(AseAtomsAdaptor.get_structure(saved_atoms))
print("Saved interface:", INTERFACE_PATH)
print("Interface atoms:", len(interface))
print(
    "Matched top/bottom slab atoms:",
    len(interface.get_centered_top_slab()),
    len(interface.get_centered_bottom_slab()),
)


## 8. Inspect the matched slabs and interface in 3D (optional)

This step uses the viewer from step 4 to show the matched slabs as `2×2×1` periodic copies and the interface in its saved unit cell. These views show the alignment and separation before geometry optimisation. They do not change the saved structure.

In [ ]:
show_structure(
    interface.get_centered_top_slab(),
    "Matched top slab (2x2x1)",
    repeat=(2, 2, 1),
)
show_structure(
    interface.get_centered_bottom_slab(),
    "Matched bottom slab (2x2x1)",
    repeat=(2, 2, 1),
)
show_structure(interface, "Assembled interface")


## Next step: optimise the interface geometry

Open [Notebook 05](05_geometry_optimisation.ipynb). Set its `INTERFACE_PATH` to `examples/results/interface.extxyz` under the repository root to use this output. Keep the interface metadata in the file. The saved structure is an unrelaxed starting geometry, not a validated interface.